# Curvature Comparative

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path


def plot_combined_curvatures(root_dir, save_path=None):
    plt.rcdefaults()
    sns.set_theme(style="white")

    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.edgecolor": "black",
        "axes.linewidth": 0.8
    })

    data_list = []
    root = Path(root_dir)
    series_order = ["shells_shifted", "cylinder_diagonal", "dumbbell_bridge"]

    for folder in root.iterdir():
        if not folder.is_dir() or folder.name.startswith('.'): continue
        series_name = folder.name
        for csv_file in folder.glob("*.csv"):
            if "curv_r" not in csv_file.name: continue
            c_type = "Gaussian" if "gauss" in csv_file.name else "Mean"
            layer = "Inner" if "inner" in csv_file.name else "Outer"
            df_temp = pd.read_csv(csv_file)
            values = df_temp.iloc[:, 0].values
            data_list.append(pd.DataFrame({
                "Series": [series_name] * len(values),
                "Layer": [layer] * len(values),
                "Type": [c_type] * len(values),
                "Curvature": values
            }))

    df = pd.concat(data_list, ignore_index=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
    palette = {"Outer": "#2c7bb6", "Inner": "#d7191c"}

    labels = {
        "Gaussian": r"Gaussian Curvature ($\mu m^{-2}$)",
        "Mean": r"Mean Curvature ($\mu m^{-1}$)"
    }

    for i, t in enumerate(["Mean", "Gaussian"]):
        ax = axes[i]
        subset = df[df["Type"] == t]
        current_order = [s for s in series_order if s in subset['Series'].unique()]

        sns.violinplot(
            data=subset, x="Series", y="Curvature", hue="Layer",
            hue_order=["Inner", "Outer"],
            order=current_order,
            ax=ax, palette=palette, split=False, inner="quartile",
            cut=0, linewidth=1.0, density_norm="width"
        )

        ax.axhline(0, color='black', linestyle='--', linewidth=0.7, alpha=0.8, zorder=0)
        ax.set_ylabel(labels[t])
        ax.set_xlabel("")
        new_labels = [n.get_text().replace('_', '\n') for n in ax.get_xticklabels()]
        ax.set_xticks(range(len(new_labels)))
        ax.set_xticklabels(new_labels)

        sns.despine(ax=ax, offset=5)
        ax.grid(axis='y', linestyle='-', color='#eeeeee', linewidth=0.5, zorder=-1)

        if ax.get_legend():
            ax.get_legend().remove()

    handles, labels_lg = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels_lg, title="Layer", frameon=True,
                   loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=2)

    plt.tight_layout()

    if save_path:
        plt.savefig(os.path.join(save_path, "combined_curvatures_final.pdf"), bbox_inches='tight', dpi=300,
                    pad_inches=0.1)

    plt.show()


root_path = "/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_CURVATURES"
plot_combined_curvatures(root_path, save_path=root_path)

# Thickness + Field

In [ ]:
from module_scripts.analysis import spherical_project, spherical_project_vectors
from module_scripts.visuals import plot_spherical_projection
from module_scripts.datahandler import load_mesh, load_array

In [ ]:
heatmappath = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted'
layer_mesh = load_mesh(
    filepath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/layer_mesh.ply')
directors_2dcurved_avg = load_array(name="directors-avg_2dcurved_r-20.0um",
                                    folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/')

idxs_sel = load_array("calcindeces",
                      folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/').astype(
    int)
thickness_vals = load_array("inner_mesh_smooth_subset_VS_outer_mesh_smooth_subset_thickness",
                            folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/')
s_2dcurv = load_array("S-order_2dcurved_r-20.0um",
                      folderpath='/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/_THICKNESS/shells_shifted/inner_mesh_smooth_subset_proj_0_to_9_um_mean/')

sph_proj_phi, sph_proj_theta = spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                       directors_2dcurved_avg[:, 3:])
plot_spherical_projection(phi=sph_proj_phi, theta=sph_proj_theta, intensities=thickness_vals, cmap="coolwarm",
                          vec_pos_phi=sph_proj_phi[idxs_sel], vec_pos_theta=sph_proj_theta[idxs_sel],
                          vec_dir_phi=vec_dir_phi, vec_dir_theta=vec_dir_theta, veccolor="k", alpha=1.0,
                          scale_factor=5, arrow_alpha=0.8, vec_manual_vminmax=[0, 1], vec_width=0.002,
                          cmap_label=r"Thickness $d$ ($\mu$m)",
                          savefig=os.path.join(heatmappath, f"spherical_projection_field_avg-nematic.pdf"),
                          figsize=(16, 6))

# RENAME previous convention NEMO layer folders

In [ ]:
import os

root_dir = '/Volumes/roux/AurelienRouxLab/Oriol/Experiments 2 photon with filter/Gastruloids size exp/EXP4_FILTER_CLEAN/'

print(">> STARTING FOLDER RENAMING...")

for root, dirs, files in os.walk(root_dir):
    for dir_name in dirs:
        # Identify target folders
        if dir_name.startswith('proj_'):

            # Construct the new folder name
            new_name = f"sampling_mesh_{dir_name}_mean"

            # Create full absolute paths
            old_path = os.path.join(root, dir_name)
            new_path = os.path.join(root, new_name)

            # Perform the rename
            try:
                os.rename(old_path, new_path)
                print(f"Renamed: {dir_name} -> {new_name}")
            except OSError as e:
                print(f"Error renaming {dir_name}: {e}")

print(">> Renaming Complete.")